# OntoHajj – LLM-based query understanding (S4-LLM-L2)
Runs S4 with LLM-based intent typing and entity extraction (config C4llm) on the 50 human-written questions and on the stratified subset of 100 template test questions, with Qwen2.5-7B on a free Colab GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1) install and start Ollama, download the model (about 5 GB, 3-5 minutes)
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
!ollama pull qwen2.5:7b

In [ ]:
# 2) get the code and data
!git clone -q https://github.com/majidphd/OntoHajj
%cd OntoHajj
!pip -q install -r requirements.txt

In [ ]:
# 3) settings: local Ollama server, same as on the laptop
import os
os.environ.update({'ONTOHAJJ_PROVIDER': 'openai', 'ONTOHAJJ_BASE_URL': 'http://localhost:11434/v1',
                   'ONTOHAJJ_API_KEY': 'ollama', 'ONTOHAJJ_NO_THINK': '0'})
!python eval/test_connection.py qwen2.5:7b

In [ ]:
# 4) the runs (about 30-45 minutes on a T4)
!python eval/run_llm_eval.py --model qwen2.5:7b --data hajjnavqa_h.jsonl --split human --configs C4llm
!python eval/run_llm_eval.py --model qwen2.5:7b --split test --limit 100 --configs C4llm --version llml2

In [ ]:
# 5) summary
!python eval/summarise_llm.py qwen2.5_7b/human_v2 | tail -4
!python eval/summarise_llm.py qwen2.5_7b/test_llml2 | tail -2

In [ ]:
# 6) download the results and send them
!cd results/llm/qwen2.5_7b && zip -qr /content/llm_l2.zip human_v2 test_llml2
from google.colab import files
files.download('/content/llm_l2.zip')